# CNN Architecture and Training

Train a small CNN to distinguish synthetic vertical and horizontal lines. The held-out split uses separately generated examples and no downloads.

In [ ]:
import torch
from torch import nn

torch.manual_seed(34)
torch.set_num_threads(1)
print('CPU | local synthetic image task')

In [ ]:
def make_images(count, seed):
    generator = torch.Generator().manual_seed(seed)
    images = torch.zeros(count, 1, 8, 8)
    labels = torch.randint(0, 2, (count,), generator=generator)
    for index, label in enumerate(labels):
        location = int(torch.randint(1, 7, (), generator=generator))
        if label == 0:
            images[index, 0, :, location] = 1
        else:
            images[index, 0, location, :] = 1
    images += 0.05 * torch.randn(images.shape, generator=generator)
    return images, labels

train_x, train_y = make_images(160, 340)
test_x, test_y = make_images(60, 341)
assert train_x.shape == (160, 1, 8, 8)
print('train/test:', train_x.shape, test_x.shape)

In [ ]:
model = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                      nn.Conv2d(8, 12, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
                      nn.Flatten(), nn.Linear(12, 2))
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
for _ in range(35):
    optimizer.zero_grad()
    loss = nn.functional.cross_entropy(model(train_x), train_y)
    loss.backward()
    optimizer.step()
with torch.no_grad():
    accuracy = (model(test_x).argmax(1) == test_y).float().mean().item()
assert torch.isfinite(loss)
print(f'train loss={loss.item():.4f} | held-out accuracy={accuracy:.1%}')